# Notebook 1: Backbone and `z` Training

This notebook trains or reloads the supervised CIFAR-10 backbone, then trains the canonical frozen-backbone `z` branch and the experimental joint-backbone `z` branch.

The key compute-saving design is that Phase C is run once per `lambda_traj` candidate and checkpointed at milestone epochs, so later notebooks can compare epochs without retraining.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")
IN_COLAB = "google.colab" in sys.modules

if REPO_DIR.parent.exists() and IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")

if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

try:
    from flow_circuits.training import load_yaml_config, run_backbone_and_z_training_workflow
except ModuleNotFoundError:
    REPO_DIR = Path("/content/model_interpretability")
    if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(Path.cwd())], check=True)
    from flow_circuits.training import load_yaml_config, run_backbone_and_z_training_workflow


## Parameters

Everything you should normally edit is in the next cell. There are no hidden run modes.


In [ ]:
CONFIG_NAME = "resnet18_z_workflow"
CONFIG_PATH = Path("configs/flow/resnet18_aligned.yaml")
BACKBONE_EPOCHS = 20
PHASE_A_EPOCHS = 20
PHASE_B_EPOCHS = 20
PHASE_C_MAX_EPOCHS = 40
PHASE_C_MILESTONES = [5, 10, 20, 40]
LAMBDA_TRAJ_CANDIDATES = [0.05, 0.1, 0.2, 0.4]
JOINT_BRANCH_ENABLED = True
JOINT_BACKBONE_LR_MULTIPLIER = 0.1
JOINT_CE_WEIGHT = 1.0
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits") if Path("/content/drive/MyDrive").exists() else Path("artifacts/flow_circuits")
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb01_backbone_and_z_training" / CONFIG_NAME


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base_config = load_yaml_config(CONFIG_PATH)
training_candidates_path = OUTPUT_DIR / "training_candidates.json"

if training_candidates_path.exists() and not FORCE_RERUN:
    RESULTS = json.loads(training_candidates_path.read_text(encoding="utf-8"))
else:
    RESULTS = run_backbone_and_z_training_workflow(
        base_config,
        backbone_epochs=BACKBONE_EPOCHS,
        phase_a_epochs=PHASE_A_EPOCHS,
        phase_b_epochs=PHASE_B_EPOCHS,
        phase_c_max_epochs=PHASE_C_MAX_EPOCHS,
        phase_c_milestones=PHASE_C_MILESTONES,
        lambda_traj_candidates=LAMBDA_TRAJ_CANDIDATES,
        output_dir=OUTPUT_DIR,
        joint_branch_enabled=JOINT_BRANCH_ENABLED,
        joint_backbone_lr_multiplier=JOINT_BACKBONE_LR_MULTIPLIER,
        joint_ce_weight=JOINT_CE_WEIGHT,
        force_rerun=FORCE_RERUN,
    )

print(f"Saved training candidates to: {training_candidates_path}")


In [ ]:
print("Backbone checkpoint:")
print(RESULTS["backbone_checkpoint"])
print("\nFrozen Phase B checkpoint:")
print(RESULTS["frozen_phase_b_checkpoint"])
print("\nFrozen Phase C candidates:")
for row in RESULTS["frozen_phase_c_candidates"]:
    print(f"  lambda={row['lambda_traj']:<4} epoch={row['epoch']:<3} -> {Path(row['checkpoint_path']).name}")
print("\nJoint Phase C candidates:")
for row in RESULTS["joint_phase_c_candidates"]:
    print(f"  lambda={row['lambda_traj']:<4} epoch={row['epoch']:<3} -> {Path(row['checkpoint_path']).name}")
